# User Info

In [1]:
import pandas as pd
import json
import requests
import time

## Compiles a List of Users from r/SuicideWatch

In [2]:
records = []
filepath = r'D:\Reddit\ZStandard\SuicideWatch_submissions'
with open(filepath, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        record = json.loads(line)
        records.append(record)
df = pd.DataFrame(records)

In [3]:
# Remove 'deleted' user
df = df[df['author'] != '[deleted]']
# Remove 'AutoModerator' user
df = df[df['author'] != 'AutoModerator']

In [ ]:
df['author'].value_counts

In [4]:
userArray = df['author'].unique()
userArrayTest = df['author'].unique()[:10]

In [5]:
del df

## API Setup


In [6]:
CLIENT_ID = 'ee5DffSIb70G-g7EOVuG2A'
CLIENT_SECRET = 'NRBMN3dQLyygyRHsknNa69eTWrxXpw'
auth = requests.auth.HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET)
with open(r"D:\Reddit\pw.txt", 'r') as f:
    pw = f.read()
data = {
    'grant_type': 'password',
    'username': 'Progzly',
    'password': pw
}
headers = {'User-Agent': 'MyAPI/0.0.1'}
res = requests.post('https://www.reddit.com/api/v1/access_token',
                    auth=auth, data=data, headers=headers)
TOKEN_DATA = res.json()
TOKEN = TOKEN_DATA['access_token']
# headers['Authorization'] = f'bearer {TOKEN}'
headers = {**headers, **{'Authorization': f'bearer {TOKEN}'}}
headers
# requests.get('https://oauth.reddit.com/api/v1/me', headers={'User-Agent': 'MyAPI/0.0.1'}).json()
requests.get('https://oauth.reddit.com/api/v1/me', headers=headers)
#Fetching Data
# res = requests.get('https://oauth.reddit.com/r/SuicideWatch/new', 
#                     headers=headers, params={'limit': 101})

<Response [200]>

## Get User Information

In [7]:
def getInfo(user):
    url = f'https://oauth.reddit.com/user/{user}/about'
    res = requests.get(url, headers=headers)
    if res.status_code == 200:
        return res.json()
    else:
        print(f"Error fetching data for user {user}: {res.status_code}")
        return {res.status_code}

### Testing / Singular Fetch

Testing Rate Limit

In [ ]:
# for i in range(101):
#     res = requests.get('https://oauth.reddit.com/r/SuicideWatch/new', 
#                         headers=headers, params={'limit': 101})
#     print(res.status_code)
#     if res.status_code != 200:
#         break

Append Features from User Json to Dataframe

In [ ]:
# User Data will be a array of dictionaries
userData = []
user = 'Progzly'
userInfo = getInfo(user)
user_features = {
    'author': user,
    'cake_day': userInfo['data']['created_utc'],
    'karma': userInfo['data']['total_karma'],
    'post_karma': userInfo['data']['link_karma'],
    'comment_karma': userInfo['data']['comment_karma'],
    'over_18': userInfo['data']['over_18']
}
userData.append(user_features)

### Recursive Information Fetch

In [8]:
len(userArray)

294702

In [ ]:
offset = 6
for i in range(10-1-offset):
    print(f"Processing batch {i+1+offset}...")
    userArrayIter = userArray[29470*(i+offset):29470*(i+1+offset)]
    print(len(userArrayIter))
    userData = []
    nullArray = []
    for user in userArrayIter:
        failed_attempts = 0
        userInfo = getInfo(user)
        while userInfo == {429}:
            failed_attempts += 1
            if failed_attempts > 6:
                print(f"Rate limit exceeded for {user}. Skipping...")
                break
            print(f"Rate limit exceeded for {user} (Fail #{failed_attempts}). Retrying in 60 seconds...)")
            time.sleep(60)
            userInfo = getInfo(user)
        if userInfo and 'data' in userInfo:
            user_features = {
                'author': user,
                'cake_day_utc': userInfo['data'].get('created_utc', None),
                'karma': userInfo['data'].get('total_karma', None),
                'post_karma': userInfo['data'].get('link_karma', None),
                'comment_karma': userInfo['data'].get('comment_karma', None),
                'over_18': userInfo['data'].get('over_18', None)
            }
            userData.append(user_features)
        if userInfo == {404}:
            nullArray.append(user)
    userData = pd.DataFrame(userData, columns=['author', 'cake_day_utc', 'karma', 'post_karma', 'comment_karma', 'over_18'])
    nullArray = pd.DataFrame(nullArray, columns=['author'])
    userData.to_csv(f'D:/Reddit/ZStandard/userData{i+offset}.csv', mode='a', index=False, header=False)
    nullArray.to_csv(f'D:/Reddit/ZStandard/nullArray{i+offset}.csv', mode='a', index=False, header=False)
    print(f"Batch {i+1+offset} processed and saved.")
    time.sleep(60)
#

Processing batch 7...
29470
Error fetching data for user throwaway284i4owp2: 404
Error fetching data for user Larry_4054: 404
Error fetching data for user lynet_101: 404
Error fetching data for user mypornissecret: 404
Error fetching data for user 0w3nR: 404
Error fetching data for user comrade----: 404
Error fetching data for user Enough-Breadfruit: 404
Error fetching data for user peace_maker65: 404
Error fetching data for user xMaji23x: 404
Error fetching data for user ZingerBurger98: 404
Error fetching data for user tra2877: 404
Error fetching data for user YaBoiAnxiousDisaster: 404
Error fetching data for user 77011: 404
Error fetching data for user Evanhasfreeapples: 404
Error fetching data for user wokwwjskskssmsksk: 404
Error fetching data for user Fancy_Earl: 404
Error fetching data for user Milo0002000: 404
Error fetching data for user ThrowRAtabascosauce: 404
Error fetching data for user hugechungus69: 404
Error fetching data for user Rjmudcat: 404
Error fetching data for us

In [ ]:
len(nullArray)
#169m for 10605

In [ ]:
# Save Null Array to CSV
nullArray = pd.DataFrame(nullArray)
nullArray.to_csv('D:/Reddit/ZStandard/NullArray0.csv', index=False, header=False)

In [ ]:
# Count Rows in .csv
test = pd.read_csv(r'D:\Reddit\ZStandard\userData0.csv', header=None)
test.columns = ['author', 'cake_day_utc', 'karma', 'post_karma', 'comment_karma', 'over_18']
len(test)


In [ ]:
28046+1424

In [ ]:
userData['cake_day_dt'] = pd.to_datetime(userData['cake_day_utc'], unit='s')
userData